In [ ]:
# use a conda torch gpu environment
# extra deps:
%pip install sentence-transformers chromadb FlagEmbedding

Pre-Process text

In [ ]:
import csv
import os

if not os.path.exists("test_notebooks"):
    os.chdir("..")

assert os.path.exists("test_notebooks")

In [ ]:
from pathlib import Path
from typing import List


lines = Path("entropy/conf/tag_datasets/danbooru.txt").read_text("utf8").splitlines()
reader = csv.reader(lines)

rows:List[dict] = []

for i, item in enumerate(reader):
    original = item[0]
    count = item[1]
    space_tag = original.replace("_", " ")

    rows.append({
        "id": str(i),
        "original": original,
        "count": count,
        "space_tag": space_tag,
    })

rows[:5]

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
import chromadb
from tqdm import tqdm

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
model = SentenceTransformer('BAAI/bge-m3', device=device)

In [ ]:
model

In [ ]:
client = chromadb.PersistentClient(path="./database/chroma_1")

In [ ]:
if False:
    client.delete_collection(name="danbooru_tags")

collection = client.get_or_create_collection(name="danbooru_tags",metadata={"hnsw:space": "cosine"})

In [ ]:
batch_size = 512 

subset = rows[:]

for i in tqdm(range(0, len(subset), batch_size)):
    batch = subset[i : i + batch_size]
    batch_tags = [p['original'] for p in batch]
    batch_processed = [p['space_tag'] for p in batch]
    ids = [str(p['id']) for p in batch]
    
    # 生成向量 (BGE-M3 默认输出 1024 维)
    # normalize_embeddings=True 对余弦相似度检索非常重要
    with torch.no_grad():
        embeddings = model.encode(
            batch_processed, 
            batch_size=batch_size, 
            normalize_embeddings=True
        ).tolist()

    
    # 插入 ChromaDB
    collection.add(
        embeddings=embeddings,
        documents=batch_tags, # 原始带下划线的标签
        ids=ids
    )

print(f"成功导入 {collection.count()} 个标签")

In [ ]:
query_text = "华丽的服饰"
query_embedding = model.encode(query_text, normalize_embeddings=True).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=150
)

print("检索到的标签:", results['documents'])

In [ ]:
candidates = results['documents'][0]
candidates = [p.replace("_", " ") for p in candidates]

# ",".join(candidates[:20])


In [ ]:
#  pip install "transformers<5"
from FlagEmbedding import FlagReranker



# 初始化重排模型 (建议使用 v2-m3，支持多语言且性能更强)
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True) 

# 构建 [查询, 候选标签] 的对
query = query_text
pairs = [[query, candidate] for candidate in candidates]

# 计算得分 (分数越高越相关)
scores = reranker.compute_score(pairs)

# 将得分与候选标签组合并排序
reranked_results = sorted(
    zip(candidates, scores), 
    key=lambda x: x[1], 
    reverse=True
)

# 输出前 20 个最精准的结果
for tag, score in reranked_results[:50]:
    print(f"Tag: {tag}, Score: {score:.4f}")

In [ ]:
%pip show transformers

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig

In [ ]:
model_name = "jinaai/jina-reranker-v2-base-multilingual"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# 如果是在 CPU 上运行，确保不使用 fp16 并在 float32 下运行
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
# 针对 Jina v2，需要关闭 use_flash_attn 才能在 CPU 上跑
config.use_flash_attn = False

jina_reranker = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    config=config,
    trust_remote_code=True, 
    torch_dtype=torch.float32
)
jina_reranker.eval()

In [ ]:
query = query_text
# 假设这是从召回阶段获得的候选 Tag
documents = candidates

# 3. 构造输入对 [Query, Document]
sentence_pairs = [[query, doc] for doc in documents]

# 4. 推理计算得分
with torch.no_grad():
    # 使用模型的 compute_score 方法（Jina 模型特有封装）
    # 如果没有该方法，可以使用标准的 tokenizer + model(input) 流程
    scores = jina_reranker.compute_score(sentence_pairs, max_length=1024)

# 5. 排序结果
results = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)

print(f"查询: {query}")
for tag, score in results[:50]:
    print(f"得分: {score:.4f} | Tag: {tag}")